In [25]:
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

In [26]:
# Read in the data
df = pd.read_csv("../data/data.csv")

#format job titles
df['job_title'] = df['job_title'].str.replace('[^a-zA-Z0-9\\s]', '', regex=True).str.lower()
df['job_title'] = df['job_title'].str.replace(r'[\r\n\s]+', ' ', regex=True)

In [27]:
# -------------------------
# Load embedding model
# -------------------------

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5580.63it/s]


In [28]:
#create new dataframe with needed columns
structured_df = df[["job_title"]].copy()

structured_df

,job_title
0,2019 ct bauer college of business graduate mag...
1,native english teacher at epik english program...
2,aspiring human resources professional
3,people development coordinator at ryan
4,advisory board member at celal bayar university
5,aspiring human resources specialist
6,student at humber college and aspiring human r...
7,hr senior specialist
8,student at humber college and aspiring human r...
9,seeking human resources hris and generalist po...


In [ ]:
#adding job field column to structured_df

#list of jobs
job_field = [
    "human resources",
    "finance",
    "customer service",
    "information technology",
    "data science",
    "engineering",
    "education",
    "other"
]

# create new feature columns for identified job title and similarity score
field_embeddings = model.encode(job_field, normalize_embeddings=True)
job_title_embeddings = model.encode(df["job_title"].fillna("").tolist(), normalize_embeddings=True)
similarities = job_title_embeddings @ field_embeddings.T

best_match_index = similarities.argmax(axis=1)
best_match_score = similarities.max(axis=1)

structured_df["job_field"] = [job_field[i] for i in best_match_index]
structured_df

,job_title,job_field
0,2019 ct bauer college of business graduate mag...,human resources
1,native english teacher at epik english program...,education
2,aspiring human resources professional,human resources
3,people development coordinator at ryan,human resources
4,advisory board member at celal bayar university,education
5,aspiring human resources specialist,human resources
6,student at humber college and aspiring human r...,human resources
7,hr senior specialist,human resources
8,student at humber college and aspiring human r...,human resources
9,seeking human resources hris and generalist po...,human resources


In [30]:
# address location column to structured_df

df["location"] = df["location"].str.replace(r"Kanada$", "Canada", regex=True).str.strip()
has_comma = df["location"].str.contains(",", na=False)

split_location = (df.loc[has_comma, "location"].str.split(",", n=1, expand=True))

df["city"] = pd.NA
df["region"] = pd.NA

df.loc[has_comma, "city"] = split_location[0].str.strip()
df.loc[has_comma, "region"] = split_location[1].str.strip()

is_region = ~df["location"].str.contains(",", na=False)
df.loc[is_region, "region"] = df.loc[is_region, "location"]


structured_df["city"] = df["city"]
structured_df["region"] = df["region"]


In [31]:
# add seniority column to structured_df
#identifying seniority levels in job titles
seniority_levels = ["Junior", "Mid", "Senior"]

senior_titles = [
    "senior professional",
    "experienced professional",
    "lead professional",
    "team lead",
    "manager",
    "senior manager",
    "director",
    "principal",
    "head of department",
    "executive"
]

junior_titles = [
    "junior",
    "junior professional",
    "intern",
    "student",
    "entry level",
    "entry level professional",
    "graduate",
    "recent graduate",
    "undergraduate",
    "undergraduate student",
    "undergraduate researcher",
    "trainee",
    "aspiring",
    "assisstant",
    "aspiring professional",    
]

senior_embeddings = model.encode(senior_titles, normalize_embeddings=True)
junior_embeddings = model.encode(junior_titles, normalize_embeddings=True)

job_embeddings = model.encode(df["job_title"].fillna("").tolist(), normalize_embeddings=True)

senior_scores = job_embeddings @ senior_embeddings.T
junior_scores = job_embeddings @ junior_embeddings.T

senior_similarity = senior_scores.max(axis=1)
junior_similarity = junior_scores.max(axis=1)

#add connections as considerations for seniority
df["connections"] = (df["connection"].str.replace("+", "", regex=False).astype("Int64"))
senior_connection_bonus = np.select([df["connections"] >= 500, df["connections"] >= 200], [0.08, 0.02], default=0)
junior_connection_bonus = np.select([df["connections"] <= 100, df["connections"] < 500], [0.08, 0.02], default=0)


adjusted_senior_similarity = senior_similarity + senior_connection_bonus
adjusted_junior_similarity = junior_similarity + junior_connection_bonus

df["adjusted_senior_similarity"] = adjusted_senior_similarity
df["adjusted_junior_similarity"] = adjusted_junior_similarity
threshold = 0.50

structured_df["seniority"] = np.select(
    [
        (adjusted_senior_similarity >= threshold) & (adjusted_senior_similarity > adjusted_junior_similarity),
        (adjusted_junior_similarity >= threshold) & (adjusted_junior_similarity > adjusted_senior_similarity)
    ],
    ["senior", "junior"],
    default="mid"
)

In [32]:
# adding target column "fit" to structured_df
structured_df["fit"] = df["fit"]

#show all columns in the structured_df
pd.set_option("display.max_rows", None)
structured_df

,job_title,job_field,city,region,seniority,fit
0,2019 ct bauer college of business graduate mag...,human resources,Houston,Texas,junior,NaN
1,native english teacher at epik english program...,education,<NA>,Canada,mid,NaN
2,aspiring human resources professional,human resources,Raleigh-Durham,North Carolina Area,junior,NaN
3,people development coordinator at ryan,human resources,Denton,Texas,mid,NaN
4,advisory board member at celal bayar university,education,İzmir,Türkiye,mid,NaN
5,aspiring human resources specialist,human resources,<NA>,Greater New York City Area,junior,NaN
6,student at humber college and aspiring human r...,human resources,<NA>,Canada,junior,NaN
7,hr senior specialist,human resources,<NA>,San Francisco Bay Area,senior,NaN
8,student at humber college and aspiring human r...,human resources,<NA>,Canada,junior,NaN
9,seeking human resources hris and generalist po...,human resources,<NA>,Greater Philadelphia Area,mid,NaN


In [38]:
#model training phase


# -------------------------
# 3. Create candidate embeddings
# -------------------------

candidate_embeddings = model.encode(structured_df["job_title"].tolist(),normalize_embeddings=True,show_progress_bar=True)


# -------------------------
# 4. Semantic search
# -------------------------

def semantic_search(query, df, candidate_embeddings, model, top_k=10):
    job_field = ["human resources","finance","customer service","information technology","data science","engineering","education","other"]

    # Embed query
    query_embedding = model.encode(query, normalize_embeddings=True)
    field_embeddings = model.encode(job_field, normalize_embeddings=True)

    # Cosine similarity
    title_similarity = candidate_embeddings @ query_embedding
    field_scores = field_embeddings @ query_embedding
    query_field = job_field[np.argmax(field_scores)]

    field_similarity =  (df["job_field"].str.lower() == query_field).astype(float)


    final_score = 0.6 * title_similarity + 0.4 * field_similarity

    # Highest similarity first
    top_indices = np.argsort(final_score)[::-1][:top_k]

    # Retrieve candidates
    results = df.iloc[top_indices].copy()

    # Add similarity score
    results["similarity"] = final_score[top_indices]

    return results

# -------------------------
# 5. Test
# -------------------------

results = semantic_search(query="Software Engineer",df=structured_df,candidate_embeddings=candidate_embeddings,model=model,top_k=10)

print(results[["job_title", "similarity"]].to_string(index=False))

Batches: 100%|██████████| 4/4 [00:00<00:00, 32.89it/s]

                                                                          job_title  similarity
information systems specialist and programmer with a love for data and organization    0.340120
                  human resources professional for the world leader in gis software    0.309537
                                            junior mes engineer information systems    0.304443
                                                       human resources professional    0.261000
                                liberal arts major aspiring human resources analyst    0.252041
                                                aspiring human resources specialist    0.240693
                                                aspiring human resources specialist    0.240693
                                                aspiring human resources specialist    0.240693
                                                aspiring human resources specialist    0.240693
                                        

In [ ]:
"""reranking

- stars one candidate
- changes weightage of the others

- for seniority, look at using an unsupervised algorithm to group into categories
"""